# cam-structural-harmony — Full Pipeline Demo

This notebook demonstrates the complete `cam_harmony` API using **synthetic data** —
no FreeSurfer installation or real MRI data required.

The flow mirrors the production pipeline:
```
Synthetic volumes  →  Intensity normalisation  →  NeuroCombat harmonisation
    →  Variance decomposition (ICC, CV, sources)  →  Figures  →  QC report
```

Steps that require external tools (SynthStrip, ROBEX, FreeSurfer recon-all) are
stubbed out with synthetic outputs so you can see expected data shapes and formats.

In [ ]:
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path

rng = np.random.default_rng(42)
TMPDIR = Path(tempfile.mkdtemp())
print('cam-structural-harmony demo — imports OK')

## Step 1 — Synthetic brain volumes

We create small 3-D arrays (32×32×32 voxels) with a realistic intensity profile:
background near zero, WM peak around 900 HU, GM around 600 HU.  
Affine is set to 3 mm isotropic so nibabel accepts them as valid NIfTI images.

In [ ]:
def make_synthetic_t1w(seed: int = 0, scanner_bias: float = 0.0) -> nib.Nifti1Image:
    """Return a 32^3 synthetic T1w with a simple WM/GM/CSF intensity model."""
    rng_local = np.random.default_rng(seed)
    vol = np.zeros((32, 32, 32), dtype=np.float32)

    # Background
    vol[:] = rng_local.normal(5, 2, vol.shape).clip(0)

    # Simulated brain region (central cube)
    brain = vol[8:24, 8:24, 8:24]
    brain[:] = rng_local.normal(600, 80, brain.shape).clip(100)   # GM

    # WM core
    wm = vol[12:20, 12:20, 12:20]
    wm[:] = rng_local.normal(900 + scanner_bias, 60, wm.shape).clip(300)

    affine = np.diag([3.0, 3.0, 3.0, 1.0])
    return nib.Nifti1Image(vol, affine)

N_SUBJECTS = 10
# Simulate two scanners with a 15% intensity offset
scanner_biases = [0.0] * 5 + [135.0] * 5   # subjects 0-4: Scanner A; 5-9: Scanner B

imgs = [make_synthetic_t1w(seed=i, scanner_bias=b) for i, b in enumerate(scanner_biases)]
img_paths = []
for i, img in enumerate(imgs):
    p = TMPDIR / f'sub-{i+1:02d}_T1w.nii.gz'
    nib.save(img, p)
    img_paths.append(p)

print(f'Created {N_SUBJECTS} synthetic T1w volumes — shape {imgs[0].shape}, dtype {imgs[0].get_data_dtype()}')
print(f'Intensity range (sub-01): {imgs[0].get_fdata().min():.1f} – {imgs[0].get_fdata().max():.1f}')

## Step 2 — Intensity normalisation

We demonstrate `batch_normalize` with the `zscore` method (individual) and
`minmax` as the naive baseline.  The `nyul` and `fcm_wm` population methods
work the same way — just pass `method='nyul'` etc.

> **Note**: the `intensity-normalization` package must be installed
> (`pip install intensity-normalization`). The calls below show the exact
> API used by the production pipeline.

In [ ]:
from cam_harmony.intensity_norm import batch_normalize

norm_dir = TMPDIR / 'minmax'
norm_paths = batch_normalize(img_paths, method='minmax', output_dir=norm_dir)
print(f'\nNormalised {len(norm_paths)} images → {norm_dir}/')

# Show that normalisation reduces the scanner intensity gap
def wm_mean(img: nib.Nifti1Image) -> float:
    d = img.get_fdata()
    return float(d[12:20, 12:20, 12:20].mean())

pre_wm_A  = [wm_mean(imgs[i]) for i in range(5)]
pre_wm_B  = [wm_mean(imgs[i]) for i in range(5, 10)]
post_wm_A = [wm_mean(nib.load(norm_paths[i])) for i in range(5)]
post_wm_B = [wm_mean(nib.load(norm_paths[i])) for i in range(5, 10)]

print(f'Pre-norm WM mean  (Scanner A): {np.mean(pre_wm_A):.1f}  ± {np.std(pre_wm_A):.1f}')
print(f'Pre-norm WM mean  (Scanner B): {np.mean(pre_wm_B):.1f} ± {np.std(pre_wm_B):.1f}')
print(f'Post-norm WM mean (minmax, A): {np.mean(post_wm_A):.3f}  ± {np.std(post_wm_A):.3f}')
print(f'Post-norm WM mean (minmax, B): {np.mean(post_wm_B):.3f}  ± {np.std(post_wm_B):.3f}')

## Step 3 — Synthetic ROI volumes (FreeSurfer stand-in)

In the real pipeline, `freesurfer.extract_aparc_aseg()` returns a DataFrame
of volumes after running `recon-all`. Here we generate equivalent synthetic
data — same shape and column structure — to demonstrate downstream steps.

We model a **scanner effect** (Scanner B volumes are 8% larger than A) on top
of realistic biological variance.

In [ ]:
ROIS = [
    'hippocampus_L', 'hippocampus_R',
    'amygdala_L', 'amygdala_R',
    'entorhinal_L', 'entorhinal_R',
    'lateral_ventricle_L', 'lateral_ventricle_R',
    'thalamus_L', 'thalamus_R',
    'caudate_L', 'caudate_R',
]

# Mean and SD (mm³) drawn from FreeSurfer literature for adults aged 50–80
# Hippocampus:       Fjell et al. 2009, ~3500–4200 mm³
# Amygdala:          Saygin et al. 2017, ~1400–1800 mm³
# Entorhinal cortex: Fischl et al. 2002, ~2000–2800 mm³ per hemisphere
# Lateral ventricle: highly age-dependent at 50–80; Pfefferbaum et al. ~10–18k mm³
# Thalamus:          Bender et al. 2016, ~6500–8000 mm³
# Caudate:           Bergfield et al. 2010, ~3000–4200 mm³
ROI_PARAMS = {
    'hippocampus_L':       (3800,  380),  'hippocampus_R':       (3900,  390),
    'amygdala_L':          (1600,  180),  'amygdala_R':          (1650,  185),
    'entorhinal_L':        (2200,  280),  'entorhinal_R':        (2250,  285),
    'lateral_ventricle_L': (12000, 4500), 'lateral_ventricle_R': (11500, 4200),
    'thalamus_L':          (7200,  480),  'thalamus_R':          (7100,  470),
    'caudate_L':           (3600,  320),  'caudate_R':           (3500,  310),
}

PRISMA_EFFECT      = 0.08   # PRISMA volumes 8% larger than TRIO
SCAN_NOISE_FRAC    = 0.02   # scanner-level noise (% of SD)
SESSION_NOISE_FRAC = 0.01   # additional per-session test-retest noise

N_SUBJECTS  = 60
subject_ids = [f'sub-{i:03d}' for i in range(N_SUBJECTS)]
ages  = rng.integers(50, 80, N_SUBJECTS)
sexes = rng.choice(['M', 'F'], N_SUBJECTS)
tivs  = rng.normal(1450, 100, N_SUBJECTS)

base_vols = {
    roi: rng.normal(ROI_PARAMS[roi][0], ROI_PARAMS[roi][1], N_SUBJECTS)
    for roi in ROIS
}

rows = []
for i, subj in enumerate(subject_ids):
    for scanner in ['TRIO', 'PRISMA']:
        scanner_mult = 1.0 + PRISMA_EFFECT if scanner == 'PRISMA' else 1.0
        for session in ['ses-1', 'ses-2']:
            row = {
                'subject_id':   subj,
                'scanner':      scanner,
                'session':      session,
                'manufacturer': 'Siemens',
                'model':        'Prisma' if scanner == 'PRISMA' else 'Trio',
                'site':         'Cambridge',
                'age':          int(ages[i]),
                'sex':          sexes[i],
                'tiv':          float(tivs[i]),
            }
            for roi in ROIS:
                base     = base_vols[roi][i] * scanner_mult
                noise_sd = ROI_PARAMS[roi][1] * (SCAN_NOISE_FRAC + SESSION_NOISE_FRAC)
                row[roi] = round(max(rng.normal(base, noise_sd), 100), 1)
            rows.append(row)

roi_df = pd.DataFrame(rows)
print(f'ROI DataFrame: {N_SUBJECTS} subjects × 2 scanners × 2 sessions = {len(roi_df)} rows')
roi_df[['subject_id', 'scanner', 'session', 'age', 'sex', 'hippocampus_L']].head(4)

## Step 4 — NeuroCombat harmonisation

`run_combat` removes scanner effects while preserving biological covariates
(age, sex, TIV). Under the hood it calls `neuroCombat` from the
`neuroCombat-sklearn` package.

In [ ]:
from cam_harmony.harmonise import run_combat, compute_combat_residuals

# Average sessions per (subject, scanner) — ComBat expects one row per subject per batch
roi_df_avg = (
    roi_df.groupby(
        ['subject_id', 'scanner', 'manufacturer', 'model', 'site', 'age', 'sex', 'tiv'],
        as_index=False,
    )[ROIS].mean()
)

harmonised_df = run_combat(
    data=roi_df_avg,
    batch_col='scanner',
    covariate_cols=['age', 'sex', 'tiv'],
    parametric=True,
)

# Variance-ratio metric: between-scanner variance / total variance, pre vs post
# This captures both mean shift and variance heterogeneity — unlike a simple mean diff
residuals = compute_combat_residuals(
    pre_harmonised=roi_df_avg,
    post_harmonised=harmonised_df,
    batch_col='scanner',
    roi_cols=ROIS,
)

print('\nComBat harmonisation — between-scanner variance ratio (pre → post):')
for roi, v in residuals.items():
    flag = '  ← still elevated' if v['post'] > 0.05 else ''
    print(f'  {roi:<25}  pre: {v["pre"]:.3f}  post: {v["post"]:.3f}  '
          f'reduction: {v["reduction_pct"]:.0f}%{flag}')

## Step 5 — Variance decomposition

We compute:
- **ICC** — intraclass correlation between scanners (target > 0.75)
- **CV** — coefficient of variation per scanner group
- **Variance decomposition** — fraction of total variance attributable to
  scanner, model, site, and residual sources

In [ ]:
from cam_harmony.variance import (
    run_variance_analysis, compute_icc_batch,
    compute_variance_components, detect_design,
)
import pandas as pd

var_dir = TMPDIR / 'variance_results'

results = run_variance_analysis(
    data=roi_df_avg,
    roi_cols=ROIS,
    batch_col='scanner',
    subject_col='subject_id',
    variance_sources=['scanner'],
    output_dir=var_dir,
    variant_label='synthstrip_minmax',
)

print('\nICC values (inter-scanner):')
for roi, val in results['icc'].items():
    flag = '  ← FLAGGED (< 0.75)' if val < 0.75 else ''
    print(f'  {roi:<25} {val:.2f}{flag}')

# Intra-scanner ICC (test-retest within each scanner, using sessions as raters)
icc_intra_trio   = compute_icc_batch(roi_df[roi_df.scanner == 'TRIO'],   ROIS, 'subject_id', 'session')
icc_intra_prisma = compute_icc_batch(roi_df[roi_df.scanner == 'PRISMA'], ROIS, 'subject_id', 'session')
icc_intra_mean   = ((icc_intra_trio + icc_intra_prisma) / 2).rename('ICC_intra')

print('\nICC values (intra-scanner, test-retest):')
for roi, val in icc_intra_mean.items():
    print(f'  {roi:<25} {val:.2f}')

# Three-way variance decomposition: between-subject, between-scanner, within-scanner
vc_df = compute_variance_components(roi_df, ROIS, 'subject_id', 'scanner')
print('\nVariance components (mean across ROIs, %):')
for col in vc_df.columns:
    print(f'  {col:<20} {vc_df[col].mean():.1f}')

# Auto-detect study design for conditional figure selection
design = detect_design(roi_df, 'subject_id', 'scanner',
                       manufacturer_col='manufacturer', session_col='session')
print(f'\nDetected design: {design}')

# Sequential ANOVA variance decomposition
vd_df = pd.read_csv(var_dir / 'variance_decomposition.csv', index_col=0)
print('\nVariance decomposition (sequential ANOVA, mean across ROIs, %):')
for col in vd_df.columns:
    print(f'  {col:<30} {vd_df[col].mean():.1f}')

## Step 6 — Figures

The plotting module generates publication-ready comparison figures. Here we
show the ICC distribution and the variance decomposition chart inline.

In [ ]:
from cam_harmony.plotting import (
    plot_icc_comparison, plot_variance_decomposition, plot_cv_heatmap,
    plot_icc_pre_post, plot_normalized_scanner_agreement,
)
from IPython.display import Image, display

fig_dir = TMPDIR / 'figures'
fig_dir.mkdir(exist_ok=True)

In [ ]:
# Inter-scanner ICC by ROI
plot_icc_comparison(
    icc_results={'synthstrip_minmax': results['icc']},
    output_path=fig_dir / 'icc_comparison.png', threshold=0.75)
display(Image(fig_dir / 'icc_comparison.png'))

In [ ]:
# CoV heatmap (scanner × ROI)
cv_df_plot = pd.DataFrame(results['cv_by_scanner']).T
plot_cv_heatmap(cv_data=cv_df_plot, output_path=fig_dir / 'cv_heatmap.png')
display(Image(fig_dir / 'cv_heatmap.png'))

In [ ]:
# Variance decomposition (sequential ANOVA)
plot_variance_decomposition(vd_df=vd_df, output_path=fig_dir / 'variance_decomposition.png')
display(Image(fig_dir / 'variance_decomposition.png'))

In [ ]:
# Intra vs inter-scanner ICC (only for paired designs with repeated sessions)
if design['is_paired'] and design['has_sessions']:
    plot_icc_pre_post(
        icc_pre=icc_intra_mean.to_dict(),
        icc_post=results['icc'],
        output_path=fig_dir / 'icc_intra_vs_inter.png',
        title='Intra-scanner (test-retest) vs inter-scanner ICC',
        label_pre='Intra-scanner (test-retest)',
        label_post='Inter-scanner',
    )
    display(Image(fig_dir / 'icc_intra_vs_inter.png'))

In [ ]:
# Normalised scanner agreement scatter (TRIO vs PRISMA, all ROIs, colour = subject)
if design['is_paired']:
    plot_normalized_scanner_agreement(
        data=roi_df_avg, roi_cols=ROIS, scanner_col='scanner',
        scanners=('TRIO', 'PRISMA'), subject_col='subject_id',
        output_path=fig_dir / 'scanner_agreement.png',
    )
    display(Image(fig_dir / 'scanner_agreement.png'))

In [ ]:
# ICC before vs after NeuroCombat harmonisation
harmonised_icc = compute_icc_batch(harmonised_df, ROIS, 'subject_id', 'scanner')
plot_icc_pre_post(
    icc_pre=results['icc'], icc_post=harmonised_icc.to_dict(),
    output_path=fig_dir / 'icc_pre_post_combat.png',
    title='ICC before and after NeuroCombat harmonisation',
    label_pre='Pre-harmonisation',
    label_post='Post-harmonisation',
)
display(Image(fig_dir / 'icc_pre_post_combat.png'))

In [ ]:
# Brain parcellation rendering — ICC and CoV mapped onto Harvard-Oxford subcortical atlas
from cam_harmony.plotting import plot_brain_metric

brain_icc_path = fig_dir / 'brain_icc.png'
plot_brain_metric(
    metric_dict=results['icc'],
    output_path=brain_icc_path,
    metric_name='ICC', vmin=0.5, vmax=1.0, cmap='RdYlGn',
    title='Inter-scanner ICC (TRIO vs PRISMA) — subcortical ROIs',
)
display(Image(brain_icc_path))

cv_by_scanner = results.get('cv_by_scanner', {})
if cv_by_scanner:
    roi_cov = {
        roi: float(sum(cv_by_scanner[roi].values()) / len(cv_by_scanner[roi]))
        for roi in cv_by_scanner if cv_by_scanner[roi]
    }
    if roi_cov:
        brain_cov_path = fig_dir / 'brain_cov.png'
        plot_brain_metric(
            metric_dict=roi_cov, output_path=brain_cov_path,
            metric_name='CoV (%)', cmap='RdYlGn_r',
            title='Coefficient of Variation (%) — subcortical ROIs',
        )
        display(Image(brain_cov_path))

## Step 7 — AI-assisted QC report

`generate_qc_report` sends the structured results to Claude and gets back a
narrative markdown report. The cell below shows the prompt structure and an
example of the output format.

> **To run this cell** you need `ANTHROPIC_API_KEY` set in your environment.  
> The example output below is representative of what Claude returns.

In [ ]:
import json
from cam_harmony.qc_assistant import build_prompt, load_variance_results

# Build mock results dict (as if loaded from JSON files)
mock_results = {
    'pipeline_variants': [
        {'label': 'synthstrip_minmax', 'skull_strip': 'synthstrip', 'norm': 'minmax'},
        {'label': 'synthstrip_zscore', 'skull_strip': 'synthstrip', 'norm': 'zscore'},
        {'label': 'robex_fcm_wm',      'skull_strip': 'robex',       'norm': 'fcm_wm'},
    ],
    'icc_results': results['icc'],
    'cv_by_scanner': {roi: {'TRIO':   round(float(roi_df[roi_df.scanner == 'TRIO'][roi].std() /
                                                   roi_df[roi_df.scanner == 'TRIO'][roi].mean() * 100), 2),
                             'PRISMA': round(float(roi_df[roi_df.scanner == 'PRISMA'][roi].std() /
                                                   roi_df[roi_df.scanner == 'PRISMA'][roi].mean() * 100), 2)}
                       for roi in ROIS},
    'combat_residuals': {roi: {'pre': 0.12, 'post': 0.03, 'reduction_pct': 75.0}
                          for roi in ROIS},
}

prompt = build_prompt(mock_results, focus_rois=['hippocampus_L', 'entorhinal_L'])
print('Prompt preview (first 400 chars):')
print('-' * 40)
print(prompt[:400] + '...')
print('-' * 40)

In [ ]:
# Example output (no API call needed to view this)
example_report = """## Executive summary

The pipeline tested 3 variants across 2 scanners (Siemens Trio → Prisma).
NeuroCombat harmonisation achieved a mean 75% reduction in between-scanner
variance. Overall ICC values are acceptable (median 0.81), with two ROIs
falling below the 0.75 threshold.

## Best performing pipeline variant

**synthstrip_zscore** achieves the best balance of ICC reliability and scanner
effect removal. The SynthStrip skull strip is more consistent across the
Trio/Prisma acquisition difference, and z-score normalisation reduces WM peak
variability before FreeSurfer segmentation.

## ROIs of concern

- **lateral_ventricle_L** (ICC = 0.69): High biological variance interacts
  with scanner segmentation differences. Consider reviewing individual
  segmentations.
- **entorhinal_L** (ICC = 0.71): Small cortical ROI sensitive to skull-strip
  boundary placement. SynthStrip + fcm_wm normalisation may improve this.

## Harmonisation effectiveness

ComBat removed 75–89% of between-scanner variance across ROIs. Residual
scanner effect is largest in entorhinal cortex (post-ComBat CV gap: 3.1%)
which may reflect genuine biological differences requiring further
investigation.

## Recommendations

1. Use **synthstrip + zscore** as the primary pipeline variant.
2. Manually inspect lateral_ventricle and entorhinal segmentations for
   outlier subjects.
3. For longitudinal analyses, apply parametric ComBat with age + TIV
   as covariates."""

print('Example QC report output (representative — requires ANTHROPIC_API_KEY to regenerate):')
print()
print(example_report)

# To generate a real report, uncomment:
# import os; os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
# from cam_harmony.qc_assistant import generate_qc_report
# report = generate_qc_report(
#     results_dir=var_dir,
#     output_path=TMPDIR / 'qc_report.md',
#     focus_rois=['hippocampus_L', 'entorhinal_L'],
# )
# print(report)

## Multi-site, multi-manufacturer design

To demonstrate pipeline generalizability, we simulate a 3-manufacturer study
(Siemens PRISMA / GE Discovery MR750 / Philips Ingenia) with 30 subjects per site.
This is a between-subjects design — subjects are not paired across manufacturers.
ICC is not computed (requires repeated measurements); the focus is variance
decomposition across manufacturer, scanner, and site.

In [ ]:
N_MULTI = 90  # 30 subjects per manufacturer
MFR_SCANNERS = {
    'Siemens': ('PRISMA',           'Prisma',          'Cambridge'),
    'GE':      ('Discovery_MR750',  'Discovery MR750', 'Oxford'),
    'Philips': ('Ingenia',          'Ingenia',         'Edinburgh'),
}
MFR_EFFECTS = {'Siemens': 0.00, 'GE': -0.05, 'Philips': -0.03}

manufacturers_list = ['Siemens'] * 30 + ['GE'] * 30 + ['Philips'] * 30
ages_m  = rng.integers(50, 80, N_MULTI)
sexes_m = rng.choice(['M', 'F'], N_MULTI)
tivs_m  = rng.normal(1450, 100, N_MULTI)

rows_m = []
for i, mfr in enumerate(manufacturers_list):
    scanner_name, model_name, site_name = MFR_SCANNERS[mfr]
    mfr_mult = 1.0 + MFR_EFFECTS[mfr]
    row = {
        'subject_id': f'multi-sub-{i:03d}', 'scanner': scanner_name,
        'manufacturer': mfr, 'model': model_name, 'site': site_name,
        'age': int(ages_m[i]), 'sex': sexes_m[i], 'tiv': float(tivs_m[i]),
    }
    for roi in ROIS:
        mu, sigma = ROI_PARAMS[roi]
        row[roi] = round(max(rng.normal(mu * mfr_mult, sigma), 100), 1)
    rows_m.append(row)

roi_df_multi = pd.DataFrame(rows_m)
design_multi = detect_design(roi_df_multi, 'subject_id', 'scanner', manufacturer_col='manufacturer')
print(f'Multi-manufacturer DataFrame: {roi_df_multi.shape}')
print(f'Detected design: {design_multi}')
roi_df_multi[['subject_id', 'scanner', 'manufacturer', 'site', 'hippocampus_L']].head(4)

In [ ]:
harmonised_multi = run_combat(
    data=roi_df_multi, batch_col='scanner',
    covariate_cols=['age', 'sex', 'tiv'], parametric=True,
)

combat_res_multi = compute_combat_residuals(
    pre_harmonised=roi_df_multi, post_harmonised=harmonised_multi,
    batch_col='scanner', roi_cols=ROIS,
)
reductions = [v['reduction_pct'] for v in combat_res_multi.values()]
print(f'ComBat reduction: {min(reductions):.0f}–{max(reductions):.0f}% across {len(ROIS)} ROIs')

var_dir_multi = TMPDIR / 'variance_results_multi'
results_multi = run_variance_analysis(
    data=roi_df_multi, roi_cols=ROIS, batch_col='scanner',
    subject_col='subject_id', variance_sources=['manufacturer', 'scanner', 'site'],
    output_dir=var_dir_multi, variant_label='multi_manufacturer',
)

vd_df_multi = pd.read_csv(var_dir_multi / 'variance_decomposition.csv', index_col=0)
print('\nVariance decomposition (mean across ROIs, %):')
for col in vd_df_multi.columns:
    print(f'  {col:<30} {vd_df_multi[col].mean():.1f}')
print(f'\n→ ICC skipped: not a paired design')
print(f'→ Bland-Altman skipped: no paired measurements')

In [ ]:
fig_dir_multi = TMPDIR / 'figures_multi'
fig_dir_multi.mkdir(exist_ok=True)

cv_multi = pd.DataFrame(results_multi['cv_by_scanner']).T
plot_cv_heatmap(cv_data=cv_multi, output_path=fig_dir_multi / 'cv_heatmap_multi.png',
                title='CoV (%) — multi-manufacturer design')
display(Image(fig_dir_multi / 'cv_heatmap_multi.png'))

plot_variance_decomposition(vd_df=vd_df_multi,
                            output_path=fig_dir_multi / 'variance_decomp_multi.png',
                            title='Variance decomposition — multi-manufacturer')
display(Image(fig_dir_multi / 'variance_decomp_multi.png'))

## Summary

This notebook demonstrated the full `cam_harmony` API:

| Step | Function | Input | Output |
|------|----------|-------|--------|
| Skull strip | `skull_strip.skull_strip()` | T1w NIfTI | stripped + mask |
| Normalise | `intensity_norm.batch_normalize()` | NIfTI list | normalised NIfTIs |
| Harmonise | `harmonise.run_combat()` | ROI DataFrame | harmonised DataFrame |
| Variance | `variance.run_variance_analysis()` | ROI DataFrame | ICC/CV/VD JSONs |
| Figures | `plotting.generate_all_figures()` | results dir | PNG figures |
| QC report | `qc_assistant.generate_qc_report()` | results dir | Markdown report |

To run the full pipeline on a BIDS dataset:
```bash
python -m cam_harmony.run \
    --bids_dir /path/to/bids \
    --output_dir outputs/ \
    --config config.yaml
```